# Paper Results Display

This notebook loads generated artifacts from `research/outputs/` and displays paper-ready sections:

- Dataset description
- Training details
- FFNN metrics
- Latency and resource usage
- Qualitative semantic examples and maintenance reports

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, Image

PROJECT_ROOT = Path('/Users/samarthatreya/Documents/Claped/PISA-Clap')
OUTPUT_DIR = PROJECT_ROOT / 'research' / 'outputs'

paths = {
    'metrics': OUTPUT_DIR / 'metrics.json',
    'summary': OUTPUT_DIR / 'paper_summary.md',
    'qualitative': OUTPUT_DIR / 'qualitative_examples.json',
    'tsne': OUTPUT_DIR / 'tsne_scatter.png',
    'umap': OUTPUT_DIR / 'umap_scatter.png',
    'confusion': OUTPUT_DIR / 'confusion_matrix.png',
    'roc': OUTPUT_DIR / 'roc_curve.png',
    'latency': OUTPUT_DIR / 'latency_bar.png',
    'zero_shot': OUTPUT_DIR / 'zero_shot_similarity_heatmap.png',
}

def show_missing(name: str):
    display(Markdown(f"> ⚠ Missing artifact: `{name}` at `{paths[name]}`"))

print('Output directory:', OUTPUT_DIR)
print('Exists:', OUTPUT_DIR.exists())
for k, p in paths.items():
    print(f"{k:>10}: {'OK' if p.exists() else 'MISSING'}")

metrics = {}
if paths['metrics'].exists():
    metrics = json.loads(paths['metrics'].read_text(encoding='utf-8'))
else:
    show_missing('metrics')

summary_text = ''
if paths['summary'].exists():
    summary_text = paths['summary'].read_text(encoding='utf-8')
else:
    show_missing('summary')


: 

## Dataset Description and Training Details

In [ ]:
dataset = metrics.get('dataset', {})
training = metrics.get('training', {})

dataset_df = pd.DataFrame([
    {
        'normal_clips': dataset.get('normal_count', 'N/A'),
        'abnormal_clips': dataset.get('abnormal_count', 'N/A'),
        'total_clips': dataset.get('total_count', 'N/A'),
        'environment': dataset.get('environment', 'N/A'),
        'pump_type': dataset.get('pump_type', 'N/A'),
    }
])

training_df = pd.DataFrame([
    {
        'epochs': training.get('epochs', 'N/A'),
        'split': training.get('split', 'N/A'),
        'metrics_note': training.get('metrics_note', 'N/A'),
    }
])

display(Markdown('### Dataset Description'))
display(dataset_df)
display(Markdown('### Training Details'))
display(training_df)

if paths['tsne'].exists() or paths['umap'].exists():
    display(Markdown('### Embedding Separability (t-SNE / UMAP)'))
    if paths['tsne'].exists():
        display(Image(filename=str(paths['tsne'])))
    else:
        show_missing('tsne')
    if paths['umap'].exists():
        display(Image(filename=str(paths['umap'])))
    else:
        show_missing('umap')


## FFNN Metrics

In [ ]:
reliability = metrics.get('reliability', {})

ffnn_df = pd.DataFrame([
    {'metric': 'threshold', 'value': reliability.get('threshold', 'N/A')},
    {'metric': 'accuracy', 'value': reliability.get('accuracy', 'N/A')},
    {'metric': 'precision_abnormal', 'value': reliability.get('precision_abnormal', 'N/A')},
    {'metric': 'recall_abnormal_tpr', 'value': reliability.get('recall_abnormal_tpr', 'N/A')},
    {'metric': 'specificity_tnr', 'value': reliability.get('specificity_tnr', 'N/A')},
    {'metric': 'false_negative_rate', 'value': reliability.get('false_negative_rate', 'N/A')},
    {'metric': 'false_positive_rate', 'value': reliability.get('false_positive_rate', 'N/A')},
    {'metric': 'auc', 'value': reliability.get('auc', 'N/A')},
])

display(ffnn_df)

display(Markdown('### Confusion Matrix'))
if paths['confusion'].exists():
    display(Image(filename=str(paths['confusion'])))
else:
    show_missing('confusion')

display(Markdown('### ROC Curve'))
if paths['roc'].exists():
    display(Image(filename=str(paths['roc'])))
else:
    show_missing('roc')


## Latency and Resource Usage

In [ ]:
latency = metrics.get('latency', {})

latency_rows = []
for mode, stats in latency.items():
    latency_rows.append({
        'mode': mode,
        'mean_ms': stats.get('mean_ms', 'N/A'),
        'std_ms': stats.get('std_ms', 'N/A'),
        'n': stats.get('n', 'N/A'),
    })

if latency_rows:
    display(pd.DataFrame(latency_rows))
else:
    display(Markdown('> ⚠ No latency data found in `metrics.json`.'))

if paths['latency'].exists():
    display(Image(filename=str(paths['latency'])))
else:
    show_missing('latency')

if summary_text:
    resource_lines = []
    for line in summary_text.splitlines():
        if line.startswith('- Device used by torch:') or line.startswith('- Host platform:') or line.startswith('- Python:') or line.startswith('- End-to-end evaluation runtime:') or line.startswith('- Process max RSS memory:'):
            resource_lines.append(line)
    if resource_lines:
        display(Markdown('### Runtime/Resource Snippets'))
        display(Markdown('\n'.join(resource_lines)))


## Qualitative Semantic Labels and Maintenance Reports

In [ ]:
if paths['qualitative'].exists():
    qualitative = json.loads(paths['qualitative'].read_text(encoding='utf-8'))
else:
    qualitative = []
    show_missing('qualitative')

if qualitative:
    qdf = pd.DataFrame(qualitative)
else:
    qdf = pd.DataFrame([
        {
            'filename': 'N/A',
            'top_label': 'N/A',
            'top_score': 'N/A',
            'maintenance_priority': 'N/A',
            'analysis_stage': 'N/A',
            'description': 'No qualitative examples available.'
        }
    ])

columns = [
    c for c in [
        'filename',
        'top_label',
        'top_score',
        'maintenance_priority',
        'analysis_stage',
        'description',
        'error',
    ] if c in qdf.columns
]
display(qdf[columns])

display(Markdown('### Zero-Shot Similarity Heatmap'))
if paths['zero_shot'].exists():
    display(Image(filename=str(paths['zero_shot'])))
else:
    show_missing('zero_shot')


## Appendix: Full Generated Summary

In [ ]:
if summary_text:
    display(Markdown(summary_text))
else:
    show_missing('summary')
